# 第五章：Starlink VHF TT&C 流量密度估算+EDA2观测轨道分析



1. 重新运行四个事件时间与方向参数的构建流程，并从原始电压数据重新测量 Event 1 和 Event 2 的事件内外功率；
2. 重新建立 E08 的 127 阵元 OSKAR 模型，运行 137.055 MHz 波束仿真并计算完整包时段的平均相对功率增益；
3. 直接读取 Grigg et al. (2025) 的原始公开目录，重新提取覆盖 137.055 MHz 的最强粗、细通道记录；
4. 计算方向增益校正后的 Event 1 包时段平均谱流量密度，归一化到 1000 km，并与既有观测作数量级比较。


## 0. 运行配置与真正的上游输入

运行本 Notebook 前需要具备：

- Event 1、Event 2 对应的原始 `.dat` 电压文件，以及第二章解调得到的 `summary.json` 和 `frames.csv`；
- STARLINK-32838 的 SupGP 数据、E08 坐标和 127 个阵元位置；
- 已安装并可执行的 OSKAR；
- Grigg et al. (2025) 公开的原始识别目录。

`apparent_flux_summary.csv`、`four_event_direction_timeseries.csv`、`four_event_oskar_gain_summary.csv` 和文献筛选结果均由下面的单元重新生成。

In [ ]:
from pathlib import Path
import math
import os
import subprocess
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'project_config.py').is_file():
            return candidate
    raise FileNotFoundError('Run from this repository or one of its subdirectories')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from project_config import DATA_DIR, RAW_DATA_DIR, RESULTS_DIR, file_start_utc
NOTEBOOK_DIR = PROJECT_ROOT / '5_flux_density'
OUTPUT_DIR = RESULTS_DIR / '5_flux_density' / 'chapter5_flux_density'
PIPELINE_NOTEBOOK_DIR = OUTPUT_DIR / 'executed_upstream_notebooks'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

PREPARATION_NOTEBOOK = NOTEBOOK_DIR / 'prepare_oskar_and_flux_inputs.ipynb'
OSKAR_RUN_SCRIPT = NOTEBOOK_DIR / 'oskar_simulation' / 'run_oskar_beam.sh'
GRIGG_CATALOGUE_FILE = DATA_DIR / 'external' / 'eda2' / 'identifications_test_v5_starlink_3_min_final_export.csv'
APPARENT_FLUX_FILE = RESULTS_DIR / '5_flux_density' / 'apparent_flux' / 'apparent_flux_summary.csv'
OSKAR_DIRECTION_FILE = RESULTS_DIR / '5_flux_density' / 'oskar_input' / 'four_event_direction_timeseries.csv'
OSKAR_GAIN_FILE = NOTEBOOK_DIR / 'oskar_simulation' / 'results' / 'tables' / 'four_event_oskar_gain_summary.csv'
GRIGG_EXTRACTION_DIR = RESULTS_DIR / '5_flux_density' / 'grigg2025_direct_137p055_read'
GRIGG_2025_FILE = GRIGG_EXTRACTION_DIR / 'strongest_coarse_and_fine_137p055MHz.csv'

for required in (PREPARATION_NOTEBOOK, OSKAR_RUN_SCRIPT, GRIGG_CATALOGUE_FILE):
    if not required.is_file():
        raise FileNotFoundError(required)

RF_FREQUENCY_HZ = 137_055_000.0
ANALYSIS_BANDWIDTH_HZ = 41_666.666666666664
BOLTZMANN_J_PER_K = 1.380649e-23
EFFECTIVE_AREA_M2 = 218.0
REFERENCE_FREQUENCY_HZ = 150e6
REFERENCE_SKY_TEMPERATURE_K = 253.2
REFERENCE_SYSTEM_TEMPERATURE_K = 350.0
SKY_SPECTRAL_INDEX = -2.55
REFERENCE_RANGE_KM = 1000.0
OSKAR_IMAGE_SIZE = 2048

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda value: f'{value:.6g}')
print('Notebook directory:', NOTEBOOK_DIR)
print('Output directory:', OUTPUT_DIR)

## 1. 重建四个事件几何并重新测量事件功率

下面在独立内核中完整执行 `prepare_oskar_and_flux_inputs.ipynb`。该流程从原始 `.dat` 文件、解调结果、SupGP和阵元坐标出发，重新生成：

- 四个事件的时间范围和包内方位角、高度角、斜距时间序列；
- 理想阵列因子诊断结果；
- Event 1和Event 2的DDC、事件内外带内功率及主波束等效表流量密度。


In [ ]:
preparation_started_ns = time.time_ns()
executed_preparation_notebook = (
    PIPELINE_NOTEBOOK_DIR / 'prepare_oskar_and_flux_inputs.executed.ipynb'
)
prepare_command = [
    sys.executable, '-m', 'jupyter', 'nbconvert',
    '--to', 'notebook', '--execute', str(PREPARATION_NOTEBOOK),
    '--output', executed_preparation_notebook.name,
    '--output-dir', str(executed_preparation_notebook.parent),
    '--ExecutePreprocessor.timeout=1800',
]
print('Running:', ' '.join(prepare_command))
subprocess.run(prepare_command, check=True, cwd=NOTEBOOK_DIR)

for produced in (APPARENT_FLUX_FILE, OSKAR_DIRECTION_FILE):
    if not produced.is_file():
        raise FileNotFoundError(f'Upstream run did not create {produced}')
    if produced.stat().st_mtime_ns < preparation_started_ns:
        raise RuntimeError(f'Upstream output was not refreshed: {produced}')

print('Fresh apparent-flux table:', APPARENT_FLUX_FILE)
print('Fresh OSKAR direction table:', OSKAR_DIRECTION_FILE)
print('Executed upstream notebook:', executed_preparation_notebook)

## 2. 重新运行OSKAR波束模拟

该步骤调用完整的OSKAR命令行流程：根据E08的127阵元坐标写出望远镜模型和配置，计算137.055 MHz全天波束，并在四个事件完整时间范围内逐点计算相对功率响应，最后在线性功率域求各事件平均增益。`OSKAR_IMAGE_SIZE`只控制全天波束图的像素数，不影响事件方向的精确采样值。

In [ ]:
oskar_started_ns = time.time_ns()
oskar_environment = os.environ.copy()
oskar_environment['IMAGE_SIZE'] = str(OSKAR_IMAGE_SIZE)
print(f'Running OSKAR with IMAGE_SIZE={OSKAR_IMAGE_SIZE}')
subprocess.run(
    ['bash', str(OSKAR_RUN_SCRIPT)],
    check=True,
    cwd=NOTEBOOK_DIR,
    env=oskar_environment,
)
if not OSKAR_GAIN_FILE.is_file():
    raise FileNotFoundError(f'OSKAR run did not create {OSKAR_GAIN_FILE}')
if OSKAR_GAIN_FILE.stat().st_mtime_ns < oskar_started_ns:
    raise RuntimeError(f'OSKAR output was not refreshed: {OSKAR_GAIN_FILE}')
print('Fresh OSKAR gain table:', OSKAR_GAIN_FILE)

## 3. 从Grigg et al. (2025)原始目录提取137.055 MHz对照记录

这里不读取先前筛选出的两行结果，而是重新分块扫描公开原始目录。分别判断31个可用细通道及其堆叠粗通道的频率边界是否覆盖137.055 MHz，再从两类记录中各选取流量密度最高的一条。

In [ ]:
# 功能：从Grigg et al. (2025)公开CSV中提取频带覆盖137.055 MHz的全部记录。
# 输出：数据行数、最强粗通道记录、最强细通道记录，以及最强细通道是否覆盖137.055 MHz。
# 保存：所有相关行CSV和两条最强记录CSV，便于后续复核。

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

CATALOGUE_CSV = GRIGG_CATALOGUE_FILE
GRIGG_EXTRACTION_DIR.mkdir(parents=True, exist_ok=True)

TARGET_FREQUENCY_HZ = 137.055e6
FINE_CHANNEL_WIDTH_HZ = 28_937.5
N_USABLE_FINE_CHANNELS = 31
STACKED_CHANNEL_INDEX = 31

COLUMN_NAMES = [
    'timestamp_unix_s', 'image_index', 'coarse_centre_hz', 'norad_id',
    'fine_channel_index', 'polarisation_index', 'flux_density_jy_beam',
    'tle_range_km', 'fit_ra_deg', 'fit_dec_deg', 'tle_ra_deg',
    'tle_dec_deg', 'fit_azimuth_deg', 'fit_elevation_deg',
    'fit_x_pixel', 'fit_y_pixel', 'tle_x_pixel', 'tle_y_pixel',
    'fit_tle_separation_pixel', 'fit_x_uncertainty_pixel',
    'fit_y_uncertainty_pixel',
]

if not CATALOGUE_CSV.is_file():
    raise FileNotFoundError(f'缺少2025年公开目录：{CATALOGUE_CSV}')

selected_chunks = []
for chunk in pd.read_csv(
    CATALOGUE_CSV,
    sep=r'\s+',
    header=None,
    names=COLUMN_NAMES,
    chunksize=1_000_000,
):
    channel_index = chunk['fine_channel_index'].astype(int)
    is_stacked = channel_index.eq(STACKED_CHANNEL_INDEX)
    is_fine = channel_index.between(0, N_USABLE_FINE_CHANNELS - 1)

    # 丢弃原始32通道中第一通道后，CSV索引0对应偏移-14.5个细通道。
    fine_centre_hz = (
        chunk['coarse_centre_hz']
        + (channel_index - 14.5) * FINE_CHANNEL_WIDTH_HZ
    )
    fine_lower_hz = fine_centre_hz - FINE_CHANNEL_WIDTH_HZ / 2
    fine_upper_hz = fine_centre_hz + FINE_CHANNEL_WIDTH_HZ / 2

    # 31个可用细通道的堆叠频带：索引0的下边界到索引30的上边界。
    stacked_lower_hz = (
        chunk['coarse_centre_hz'] - 15.0 * FINE_CHANNEL_WIDTH_HZ
    )
    stacked_upper_hz = (
        chunk['coarse_centre_hz'] + 16.0 * FINE_CHANNEL_WIDTH_HZ
    )

    fine_contains_target = (
        is_fine
        & fine_lower_hz.le(TARGET_FREQUENCY_HZ)
        & fine_upper_hz.ge(TARGET_FREQUENCY_HZ)
    )
    stacked_contains_target = (
        is_stacked
        & stacked_lower_hz.le(TARGET_FREQUENCY_HZ)
        & stacked_upper_hz.ge(TARGET_FREQUENCY_HZ)
    )
    keep = fine_contains_target | stacked_contains_target

    if keep.any():
        selected = chunk.loc[keep].copy()
        selected['utc'] = pd.to_datetime(
            selected['timestamp_unix_s'], unit='s', utc=True
        )
        selected['data_product'] = np.where(
            selected['fine_channel_index'].astype(int).eq(STACKED_CHANNEL_INDEX),
            '0.90 MHz stacked coarse image',
            '28.9375 kHz fine-channel image',
        )
        selected['measurement_centre_hz'] = np.where(
            selected['fine_channel_index'].astype(int).eq(STACKED_CHANNEL_INDEX),
            selected['coarse_centre_hz'],
            fine_centre_hz.loc[keep],
        )
        selected['measurement_lower_hz'] = np.where(
            selected['fine_channel_index'].astype(int).eq(STACKED_CHANNEL_INDEX),
            stacked_lower_hz.loc[keep],
            fine_lower_hz.loc[keep],
        )
        selected['measurement_upper_hz'] = np.where(
            selected['fine_channel_index'].astype(int).eq(STACKED_CHANNEL_INDEX),
            stacked_upper_hz.loc[keep],
            fine_upper_hz.loc[keep],
        )
        selected['contains_137p055_mhz'] = (
            selected['measurement_lower_hz'].le(TARGET_FREQUENCY_HZ)
            & selected['measurement_upper_hz'].ge(TARGET_FREQUENCY_HZ)
        )
        selected['flux_density_mjy_beam'] = (
            selected['flux_density_jy_beam'] / 1e6
        )
        selected_chunks.append(selected)

if not selected_chunks:
    raise RuntimeError('CSV中没有找到测量频带覆盖137.055 MHz的记录。')

data_137p055 = pd.concat(selected_chunks, ignore_index=True)
coarse_137p055 = data_137p055.loc[
    data_137p055['fine_channel_index'].astype(int).eq(STACKED_CHANNEL_INDEX)
].copy()
fine_137p055 = data_137p055.loc[
    data_137p055['fine_channel_index'].astype(int).between(0, 30)
].copy()

strongest_coarse = coarse_137p055.loc[
    coarse_137p055['flux_density_jy_beam'].idxmax()
].copy()
strongest_fine = fine_137p055.loc[
    fine_137p055['flux_density_jy_beam'].idxmax()
].copy()

summary_columns = [
    'data_product', 'utc', 'norad_id', 'polarisation_index',
    'fine_channel_index', 'tle_range_km', 'flux_density_jy_beam',
    'flux_density_mjy_beam', 'measurement_centre_hz',
    'measurement_lower_hz', 'measurement_upper_hz',
    'contains_137p055_mhz',
]
maxima = pd.DataFrame([strongest_coarse, strongest_fine])[summary_columns]

all_output = GRIGG_EXTRACTION_DIR / 'all_measurements_covering_137p055MHz.csv'
maxima_output = GRIGG_EXTRACTION_DIR / 'strongest_coarse_and_fine_137p055MHz.csv'
data_137p055.to_csv(all_output, index=False)
maxima.to_csv(maxima_output, index=False)

print('Grigg et al. (2025) CSV直接读取137.055 MHz结果')
print(f'  频带覆盖137.055 MHz的全部记录：{len(data_137p055):,}')
print(f'  粗通道堆叠图记录：{len(coarse_137p055):,}')
print(f'  包含137.055 MHz的细通道记录：{len(fine_137p055):,}')
print()
print('最强粗通道与细通道记录：')
display(maxima)

fine_centre_mhz = strongest_fine['measurement_centre_hz'] / 1e6
fine_lower_mhz = strongest_fine['measurement_lower_hz'] / 1e6
fine_upper_mhz = strongest_fine['measurement_upper_hz'] / 1e6
print('最强细通道的频率复核：')
print(f'  细通道索引：{int(strongest_fine["fine_channel_index"])}')
print(f'  中心频率：{fine_centre_mhz:.9f} MHz')
print(f'  通道范围：{fine_lower_mhz:.9f} -- {fine_upper_mhz:.9f} MHz')
print(f'  是否包含137.055 MHz：{bool(strongest_fine["contains_137p055_mhz"])}')
print()
print(f'已保存全部相关行：{all_output}')
print(f'已保存两条最强记录：{maxima_output}')


## 4. 从事件内外功率得到主波束等效表流量密度

原始实值电压数据先在各事件的实测中心频率附近进行数字下变频，得到 240 ksample s$^{-1}$ 的复基带数据。随后使用 4096 点 Hann 窗计算互不重叠的功率谱，并在 41.667 kHz LoRa 带宽内积分。

事件窗口从 preamble start 延伸至 packet stop。背景由事件前后各 1 s 的数据共同组成，并在事件与每段背景之间保留 0.30 s 保护间隔。$P_{\rm on}$ 和 $P_{\rm off}$ 分别为事件窗口和两段背景内各功率谱带内积分值的算术平均。信号相对于系统背景的功率增量为

$$
\eta=\frac{P_{\rm on}}{P_{\rm off}}-1.
$$

采用单个 E08 pod 的 $A_{\rm eff}=218\,\mathrm{m^2}$，并将 150 MHz 下的天空温度按谱指数 $-2.55$ 缩放至 137.055 MHz。系统温度及主波束 SEFD 为

$$
T_{\rm sys}(\nu)=T_{\rm sky,150}\left(\frac{\nu}{150\,\mathrm{MHz}}\right)^{-2.55} + (T_{\rm sys,150}-T_{\rm sky,150}),
$$

$$
\mathrm{SEFD}=\frac{2k_{\rm B}T_{\rm sys}}{A_{\rm eff}}.
$$

由此得到未经事件方向旁瓣校正的主波束等效表流量密度 $S_{\nu,\rm app}=\eta\,\mathrm{SEFD}$。它表示若相同功率增量来自主波束方向，对应的等效谱流量密度。

In [ ]:
apparent_input = pd.read_csv(APPARENT_FLUX_FILE)

non_sky_temperature_k = (
    REFERENCE_SYSTEM_TEMPERATURE_K - REFERENCE_SKY_TEMPERATURE_K
)
sky_temperature_k = REFERENCE_SKY_TEMPERATURE_K * (
    RF_FREQUENCY_HZ / REFERENCE_FREQUENCY_HZ
) ** SKY_SPECTRAL_INDEX
system_temperature_k = sky_temperature_k + non_sky_temperature_k
sefd_jy = (
    2.0 * BOLTZMANN_J_PER_K * system_temperature_k
    / EFFECTIVE_AREA_M2 / 1e-26
)

apparent = apparent_input.copy()
apparent['recomputed_excess_power_ratio'] = (
    apparent['on_power_arbitrary'] / apparent['off_power_arbitrary'] - 1.0
)
apparent['recomputed_apparent_flux_density_jy'] = (
    sefd_jy * apparent['recomputed_excess_power_ratio']
)
if not np.allclose(
    apparent['recomputed_apparent_flux_density_jy'],
    apparent['nominal_main_beam_apparent_flux_density_jy'],
    rtol=1e-12, atol=1e-9,
):
    raise ValueError('Recomputed apparent flux densities do not match the saved DDC results')

apparent['apparent_flux_density_kjy'] = (
    apparent['recomputed_apparent_flux_density_jy'] / 1e3
)
apparent['packet_average_band_flux_w_m2'] = (
    apparent['recomputed_apparent_flux_density_jy']
    * 1e-26 * apparent['analysis_bandwidth_hz']
)

print(f'Sky temperature at 137.055 MHz: {sky_temperature_k:.3f} K')
print(f'System temperature at 137.055 MHz: {system_temperature_k:.3f} K')
print(f'Nominal main-beam SEFD: {sefd_jy:.3f} Jy')
display(apparent[[
    'event_id', 'packet_duration_s', 'on_power_arbitrary',
    'off_power_arbitrary', 'recomputed_excess_power_ratio',
    'apparent_flux_density_kjy', 'packet_average_band_flux_w_m2',
]])

Event 1 和 Event 2 的主波束等效表流量密度分别约为 187.7 kJy 和 5.38 kJy。表内的形式标准误差只描述有限功率谱样本造成的散布，不包含有效面积、接收链线性、极化耦合和旁瓣模型等系统误差，因此不作为最终流量密度的不确定度。

## 5. OSKAR 方向增益校正

OSKAR 模型使用 E08 的 127 个实测阵元位置，在 137.055 MHz 下以北天极为相位中心，并将相位中心的功率响应归一化为 1。每个阵元被近似为相同的各向同性、非偏振阵元，采用等权合成；模型没有包含真实阵元方向图、互耦、地面与山体、电缆延迟、增益和相位误差。

卫星方向由 SupGP 传播至完整包内的 207 个采样时刻得到。先在线性功率域对相对增益求时间平均，再转换为 dB。方向校正采用

$$
S_{\nu,\rm corr}=\frac{S_{\nu,\rm app}}{\overline{G}_{\rm rel}}.
$$

Event 2 位于模型中约 $-47$ dB 的深旁瓣方向，其校正结果会对未建模的真实波束细节极为敏感；并且理想模型无法解释两事件实测强度比。因此本文只采用处于较高响应方向的 Event 1 进行数量级估算，Event 2 仅保留未经方向校正的表流量密度。

In [ ]:
gain_table = pd.read_csv(OSKAR_GAIN_FILE)
observed_gain = gain_table.loc[
    gain_table['event_id'].isin(['Event 1', 'Event 2'])
].copy()
observed_gain['recomputed_gain_db'] = 10.0 * np.log10(
    observed_gain['mean_relative_power_linear']
)
if not np.allclose(
    observed_gain['recomputed_gain_db'],
    observed_gain['mean_relative_gain_db'],
    rtol=0.0, atol=1e-9,
):
    raise ValueError('Linear and dB OSKAR gains are inconsistent')

event1_apparent = apparent.loc[apparent['event_id'].eq('Event 1')].iloc[0]
event1_gain = gain_table.loc[gain_table['event_id'].eq('Event 1')].iloc[0]
event1_apparent_jy = float(event1_apparent['recomputed_apparent_flux_density_jy'])
event1_gain_linear = float(event1_gain['mean_relative_power_linear'])
event1_corrected_jy = event1_apparent_jy / event1_gain_linear
event1_corrected_mjy = event1_corrected_jy / 1e6
event1_band_flux_w_m2 = (
    event1_corrected_jy * 1e-26 * ANALYSIS_BANDWIDTH_HZ
)
event1_fluence_j_m2 = (
    event1_band_flux_w_m2 * float(event1_apparent['packet_duration_s'])
)

display(observed_gain[[
    'event_id', 'num_direction_samples', 'mean_relative_power_linear',
    'mean_relative_gain_db', 'mean_azimuth_deg', 'mean_elevation_deg',
    'mean_slant_range_km', 'max_abs_oskar_minus_direct_formula_db',
]])
print('Event 1 beam-corrected, packet-averaged result')
print(f'  Spectral flux density: {event1_corrected_jy:,.6g} Jy')
print(f'  Spectral flux density: {event1_corrected_mjy:.6f} MJy')
print(f'  Spectral flux density: {event1_corrected_jy * 1e-26:.6e} W m^-2 Hz^-1')
print(f'  Band-integrated average flux: {event1_band_flux_w_m2:.6e} W m^-2')
print(f'  Packet fluence: {event1_fluence_j_m2:.6e} J m^-2')

## 6. 归一化至 1000 km 并保存主结果

Event 1 的 $S_{\nu,\rm corr}$ 是信号传播到 21CMA 处的入射谱流量密度。为便于不同观测几何之间比较，采用平方反比关系将其归一化到 $R_0=1000$ km：

$$
S_{\nu,1000}=S_{\nu,\rm corr}\left(\frac{R_{\rm obs}}{1000\,\mathrm{km}}\right)^2.
$$

这一操作只去除了自由空间距离缩放；它假设卫星沿观测方向的等效发射强度、发射天线方向图、姿态、极化和发射状态不变，不能解释为卫星固有发射功率。

In [ ]:
event1_range_km = float(event1_gain['mean_slant_range_km'])
distance_scale = (event1_range_km / REFERENCE_RANGE_KM) ** 2
event1_1000km_jy = event1_corrected_jy * distance_scale
event1_1000km_mjy = event1_1000km_jy / 1e6

main_result = pd.DataFrame([{
    'event_id': 'Event 1',
    'packet_duration_s': float(event1_apparent['packet_duration_s']),
    'analysis_bandwidth_hz': ANALYSIS_BANDWIDTH_HZ,
    'apparent_flux_density_jy': event1_apparent_jy,
    'oskar_mean_relative_power_linear': event1_gain_linear,
    'oskar_mean_relative_gain_db': float(event1_gain['mean_relative_gain_db']),
    'mean_slant_range_km': event1_range_km,
    'beam_corrected_flux_density_jy': event1_corrected_jy,
    'beam_corrected_flux_density_mjy': event1_corrected_mjy,
    'beam_corrected_flux_density_w_m2_hz': event1_corrected_jy * 1e-26,
    'band_integrated_average_flux_w_m2': event1_band_flux_w_m2,
    'packet_fluence_j_m2': event1_fluence_j_m2,
    'reference_range_km': REFERENCE_RANGE_KM,
    'reference_range_normalised_flux_density_jy': event1_1000km_jy,
    'reference_range_normalised_flux_density_mjy': event1_1000km_mjy,
}])
main_output = OUTPUT_DIR / 'event1_flux_density_summary.csv'
main_result.to_csv(main_output, index=False)

display(main_result.T.rename(columns={0: 'value'}))
print(f'Observed-geometry result: {event1_corrected_mjy:.6f} MJy at {event1_range_km:.3f} km')
print(f'Normalised to 1000 km: {event1_1000km_mjy:.6f} MJy')
print('Saved:', main_output)

## 7. 与 Grigg et al. (2023, 2025) 的结果比较

Grigg et al. (2023) 使用 EDA2 在约 137.5 MHz 报道了约 100 s 重复的短时 Starlink 信号。其最高值约为 1 MJy beam$^{-1}$、斜距约 500 km，但该值来自 2 s 时间积分和约 0.926 MHz 频率平均；论文没有公开该最强事件的精确 UTC 和卫星编号。因此这里只给出距离归一化值，并不将其与本工作的窄带包平均值作精确比较。

Grigg et al. (2025) 的公开目录同时提供约 0.90 MHz 堆叠粗通道图和 28.9375 kHz 细通道图。覆盖 137.055 MHz 的最强细通道记录中心频率为 137.051581 MHz，通道边界为 137.037113--137.066050 MHz，确实包含本工作的 137.055 MHz 信道。对未分辨点源，图像峰值的 MJy beam$^{-1}$ 可在数值上近似与点源流量密度 MJy 比较，但两项观测的时间积分、带宽、极化、卫星几何和波束系统误差仍然不同。

为避免混合不同假设，下面首先只作 $R^{-2}$ 距离归一化。2025 年细通道记录是与本工作最接近的比较量；粗通道结果保留用于展示宽带平均造成的谱稀释。

In [ ]:
grigg_2025 = pd.read_csv(GRIGG_2025_FILE)
grigg_2025['normalised_1000km_mjy'] = (
    grigg_2025['flux_density_mjy_beam']
    * (grigg_2025['tle_range_km'] / REFERENCE_RANGE_KM) ** 2
)

fine_row = grigg_2025.loc[
    grigg_2025['data_product'].str.contains('fine-channel')
].iloc[0]
coarse_row = grigg_2025.loc[
    grigg_2025['data_product'].str.contains('stacked coarse')
].iloc[0]
grigg_2023_1000km_mjy = 1.0 * (500.0 / REFERENCE_RANGE_KM) ** 2
fine_to_event1_ratio = (
    float(fine_row['normalised_1000km_mjy']) / event1_1000km_mjy
)
fine_to_event1_db = 10.0 * math.log10(fine_to_event1_ratio)

comparison = pd.DataFrame([
    {
        'source': 'Grigg et al. (2023)',
        'measurement': 'published maximum near 137.5 MHz',
        'time_integration_s': 2.0,
        'frequency_bandwidth_khz': 926.0,
        'observed_slant_range_km': 500.0,
        'observed_flux_density_mjy': 1.0,
        'normalised_1000km_mjy': grigg_2023_1000km_mjy,
        'comparison_note': 'distance-only; strong time/frequency dilution',
    },
    {
        'source': 'Grigg et al. (2025)',
        'measurement': '0.90 MHz stacked coarse image',
        'time_integration_s': 2.0,
        'frequency_bandwidth_khz': (
            float(coarse_row['measurement_upper_hz'])
            - float(coarse_row['measurement_lower_hz'])
        ) / 1e3,
        'observed_slant_range_km': float(coarse_row['tle_range_km']),
        'observed_flux_density_mjy': float(coarse_row['flux_density_mjy_beam']),
        'normalised_1000km_mjy': float(coarse_row['normalised_1000km_mjy']),
        'comparison_note': 'distance-only; coarse-band spectral dilution',
    },
    {
        'source': 'Grigg et al. (2025)',
        'measurement': '28.9375 kHz fine-channel image',
        'time_integration_s': 2.0,
        'frequency_bandwidth_khz': (
            float(fine_row['measurement_upper_hz'])
            - float(fine_row['measurement_lower_hz'])
        ) / 1e3,
        'observed_slant_range_km': float(fine_row['tle_range_km']),
        'observed_flux_density_mjy': float(fine_row['flux_density_mjy_beam']),
        'normalised_1000km_mjy': float(fine_row['normalised_1000km_mjy']),
        'comparison_note': 'closest published comparison; still 2 s averaged',
    },
    {
        'source': 'This work',
        'measurement': 'Event 1 packet-average, OSKAR corrected',
        'time_integration_s': float(event1_apparent['packet_duration_s']),
        'frequency_bandwidth_khz': ANALYSIS_BANDWIDTH_HZ / 1e3,
        'observed_slant_range_km': event1_range_km,
        'observed_flux_density_mjy': event1_corrected_mjy,
        'normalised_1000km_mjy': event1_1000km_mjy,
        'comparison_note': 'model-dependent ideal E08 sidelobe correction',
    },
])
comparison_output = OUTPUT_DIR / 'literature_flux_density_comparison.csv'
comparison.to_csv(comparison_output, index=False)

display(comparison)
print('Grigg et al. (2025) strongest fine-channel record / Event 1')
print(f'  Ratio after 1000-km normalisation: {fine_to_event1_ratio:.3f}')
print(f'  Difference: {fine_to_event1_db:.3f} dB')
print('Saved:', comparison_output)

## 8. 仅距离归一化的初步对照

本分析得到 Event 1 在观测几何下的包时段平均谱流量密度约为 **20.6 MJy**，对应完整包期间平均斜距约 **669.5 km**。仅按平方反比关系归一化到 1000 km 后约为 **9.23 MJy**。Grigg et al. (2025) 公开目录中覆盖 137.055 MHz 的最强细通道记录仅作距离归一化后约为 **19.1 MJy beam$^{-1}$**。这是尚未校正 2 s 成像积分的中间诊断量，不作为最终对照结果。

下一节在此基础上统一加入 2023 年 4 s 平均、2025 年 2 s 平均和本文 1.027584 s 包时长的条件换算，并给出 Event 1 的 EIRP。

## 9. 统一时间尺度的文献对照与 Event 1 EIRP

本节给出最终采用的条件换算，第 7--8 节中仅作距离归一化的结果保留为中间诊断。

### 9.1 Grigg et al. (2023)

2023 年工作报告的最强 137.5 MHz 闪现约为 1 MJy beam$^{-1}$、斜距约 500 km。该数值是 XX 和 YY 两个极化校正后之和，并由两幅 2 s 图像平均得到 4 s 时间尺度，频率上平均了 0.926 MHz。若假设它与本文 Event 1 具有相同的 1.027584 s 包长和 41.667 kHz 带宽，则恢复时间和频率平均并归一化至 1000 km。由于本文 E08 为单极化，代码同时给出两极化总和值和假设两极化均分时的单极化等效值。

### 9.2 Grigg et al. (2025)

2025 年公开目录中最强窄带事件的两个相邻 28.9375 kHz 细通道分别约为 102.85 和 97.07 MJy beam$^{-1}$。细通道已经窄于本文 41.667 kHz 分析带宽，因此不再施加频率稀释修正，而是取两个强通道的平均。随后在‘包长与 Event 1 相同且完整位于一个 2 s 积分窗’的条件下作时间修正，并归一化至 1000 km。

### 9.3 Event 1 EIRP

对 Event 1，谱 EIRP 和 41.667 kHz 带宽内的带宽积分 EIRP 分别按

$$
\mathrm{EIRP}_{\nu}=4\pi R^2S_{\nu},
$$

$$
\mathrm{EIRP}=\mathrm{EIRP}_{\nu}B
$$

计算。这是沿地球观测方向推得的等效全向辐射功率，不是卫星发射机的实际功率。

In [ ]:
# 统一处理 2023/2025 文献的时间、频率与距离尺度，并计算 Event 1 EIRP。
packet_duration_s = float(event1_apparent['packet_duration_s'])

# Grigg et al. (2023): 4 s x 0.926 MHz，XX+YY 总和。
grigg2023_observed_mjy = 1.0
grigg2023_range_km = 500.0
grigg2023_integration_s = 4.0
grigg2023_bandwidth_hz = 0.926e6
grigg2023_time_factor = grigg2023_integration_s / packet_duration_s
grigg2023_bandwidth_factor = grigg2023_bandwidth_hz / ANALYSIS_BANDWIDTH_HZ
grigg2023_distance_factor = (grigg2023_range_km / REFERENCE_RANGE_KM) ** 2
grigg2023_totalpol_1000km_mjy = (
    grigg2023_observed_mjy * grigg2023_time_factor
    * grigg2023_bandwidth_factor * grigg2023_distance_factor
)
grigg2023_equal_split_1000km_mjy = grigg2023_totalpol_1000km_mjy / 2.0

# Grigg et al. (2025): 找回与最强记录同一时刻、卫星和偏振的 31 个细通道。
catalogue_columns = [
    'timestamp_unix_s', 'image_index', 'coarse_centre_hz', 'norad_id',
    'fine_channel_index', 'polarisation_index', 'flux_density_jy_beam',
    'tle_range_km', 'fit_ra_deg', 'fit_dec_deg', 'tle_ra_deg',
    'tle_dec_deg', 'fit_azimuth_deg', 'fit_elevation_deg',
    'fit_x_pixel', 'fit_y_pixel', 'tle_x_pixel', 'tle_y_pixel',
    'fit_tle_separation_pixel', 'fit_x_uncertainty_pixel',
    'fit_y_uncertainty_pixel',
]
target_timestamp = float(strongest_fine['timestamp_unix_s'])
target_norad = int(strongest_fine['norad_id'])
target_polarisation = int(strongest_fine['polarisation_index'])
target_coarse_centre_hz = float(strongest_fine['coarse_centre_hz'])
profile_chunks = []
for chunk in pd.read_csv(
    GRIGG_CATALOGUE_FILE, sep=r'\s+', header=None,
    names=catalogue_columns, chunksize=1_000_000,
):
    keep = (
        np.isclose(chunk['timestamp_unix_s'], target_timestamp, atol=1e-4)
        & chunk['norad_id'].astype(int).eq(target_norad)
        & chunk['polarisation_index'].astype(int).eq(target_polarisation)
        & np.isclose(chunk['coarse_centre_hz'], target_coarse_centre_hz, atol=1.0)
        & chunk['fine_channel_index'].between(0, 30)
    )
    if keep.any():
        profile_chunks.append(chunk.loc[keep].copy())

if not profile_chunks:
    raise RuntimeError('Could not recover the 31-channel profile of the strongest 2025 event')

grigg2025_profile = pd.concat(profile_chunks, ignore_index=True)
grigg2025_profile['fine_channel_index'] = grigg2025_profile['fine_channel_index'].astype(int)
grigg2025_profile['flux_density_mjy_beam'] = grigg2025_profile['flux_density_jy_beam'] / 1e6
grigg2025_profile['fine_channel_centre_hz'] = (
    grigg2025_profile['coarse_centre_hz']
    + (grigg2025_profile['fine_channel_index'] - 14.5) * 28_937.5
)
grigg2025_profile = grigg2025_profile.sort_values('fine_channel_index')
grigg2025_top_two = grigg2025_profile.nlargest(2, 'flux_density_jy_beam').sort_values('fine_channel_index')
grigg2025_two_channel_mean_mjy = float(grigg2025_top_two['flux_density_mjy_beam'].mean())
grigg2025_range_km = float(grigg2025_top_two['tle_range_km'].iloc[0])
grigg2025_integration_s = 2.0
grigg2025_time_factor = grigg2025_integration_s / packet_duration_s
grigg2025_distance_factor = (grigg2025_range_km / REFERENCE_RANGE_KM) ** 2
grigg2025_1000km_mjy = (
    grigg2025_two_channel_mean_mjy
    * grigg2025_time_factor * grigg2025_distance_factor
)
profile_output = OUTPUT_DIR / 'grigg2025_strongest_event_fine_channel_profile.csv'
grigg2025_profile.to_csv(profile_output, index=False)

# 最终比较表。
final_comparison = pd.DataFrame([
    {
        'source': 'Grigg et al. (2023), XX+YY total',
        'observed_flux_density_mjy': grigg2023_observed_mjy,
        'time_correction_factor': grigg2023_time_factor,
        'bandwidth_correction_factor': grigg2023_bandwidth_factor,
        'distance_correction_factor': grigg2023_distance_factor,
        'corrected_1000km_mjy': grigg2023_totalpol_1000km_mjy,
        'note': 'assumes Event 1 duration and bandwidth; summed two polarisations',
    },
    {
        'source': 'Grigg et al. (2023), equal-split single-pol equivalent',
        'observed_flux_density_mjy': grigg2023_observed_mjy / 2.0,
        'time_correction_factor': grigg2023_time_factor,
        'bandwidth_correction_factor': grigg2023_bandwidth_factor,
        'distance_correction_factor': grigg2023_distance_factor,
        'corrected_1000km_mjy': grigg2023_equal_split_1000km_mjy,
        'note': 'additional assumption of equal power in XX and YY',
    },
    {
        'source': 'Grigg et al. (2025), mean of two strongest fine channels',
        'observed_flux_density_mjy': grigg2025_two_channel_mean_mjy,
        'time_correction_factor': grigg2025_time_factor,
        'bandwidth_correction_factor': 1.0,
        'distance_correction_factor': grigg2025_distance_factor,
        'corrected_1000km_mjy': grigg2025_1000km_mjy,
        'note': 'assumes Event 1 duration and full containment in one 2-s image',
    },
    {
        'source': 'This work, Event 1',
        'observed_flux_density_mjy': event1_corrected_mjy,
        'time_correction_factor': 1.0,
        'bandwidth_correction_factor': 1.0,
        'distance_correction_factor': distance_scale,
        'corrected_1000km_mjy': event1_1000km_mjy,
        'note': 'packet-average; model-dependent ideal E08 sidelobe correction',
    },
])
final_comparison['ratio_to_event1'] = final_comparison['corrected_1000km_mjy'] / event1_1000km_mjy
final_comparison['difference_from_event1_db'] = 10.0 * np.log10(final_comparison['ratio_to_event1'])
final_comparison_output = OUTPUT_DIR / 'literature_comparison_time_bandwidth_distance_corrected.csv'
final_comparison.to_csv(final_comparison_output, index=False)

# Event 1 EIRP。
event1_range_m = event1_range_km * 1e3
event1_flux_w_m2_hz = event1_corrected_jy * 1e-26
event1_spectral_eirp_w_hz = 4.0 * math.pi * event1_range_m**2 * event1_flux_w_m2_hz
event1_band_eirp_w = event1_spectral_eirp_w_hz * ANALYSIS_BANDWIDTH_HZ
event1_band_eirp_dbw = 10.0 * math.log10(event1_band_eirp_w)
event1_band_eirp_dbm = event1_band_eirp_dbw + 30.0
event1_isotropic_equivalent_packet_energy_j = event1_band_eirp_w * packet_duration_s
eirp_summary = pd.DataFrame([{
    'event_id': 'Event 1',
    'mean_slant_range_km': event1_range_km,
    'packet_average_flux_density_mjy': event1_corrected_mjy,
    'analysis_bandwidth_hz': ANALYSIS_BANDWIDTH_HZ,
    'packet_duration_s': packet_duration_s,
    'spectral_eirp_w_hz': event1_spectral_eirp_w_hz,
    'band_integrated_eirp_w': event1_band_eirp_w,
    'band_integrated_eirp_mw': event1_band_eirp_w * 1e3,
    'band_integrated_eirp_dbw': event1_band_eirp_dbw,
    'band_integrated_eirp_dbm': event1_band_eirp_dbm,
    'isotropic_equivalent_packet_energy_j': event1_isotropic_equivalent_packet_energy_j,
}])
eirp_output = OUTPUT_DIR / 'event1_eirp_summary.csv'
eirp_summary.to_csv(eirp_output, index=False)

print('Grigg et al. (2025): two strongest adjacent fine channels')
display(grigg2025_top_two[[
    'timestamp_unix_s', 'norad_id', 'polarisation_index',
    'fine_channel_index', 'fine_channel_centre_hz',
    'flux_density_mjy_beam', 'tle_range_km',
]])
print('Final literature comparison')
display(final_comparison)
print('Event 1 EIRP')
display(eirp_summary.T.rename(columns={0: 'value'}))
print(f'2023 equal-split single-pol equivalent at 1000 km: {grigg2023_equal_split_1000km_mjy:.3f} MJy')
print(f'2025 time- and distance-corrected value at 1000 km: {grigg2025_1000km_mjy:.3f} MJy')
print(f'Event 1 band-integrated EIRP: {event1_band_eirp_w * 1e3:.3f} mW')
print('Saved:', profile_output)
print('Saved:', final_comparison_output)
print('Saved:', eirp_output)

### 9.4 最终解读

按照与 Event 1 相同的包长和带宽作条件换算后，2023 年最强事件在 1000 km 处约为 **21.6 MJy beam$^{-1}$**（XX+YY 总和）；若额外假设两极化均分，单极化等效值约为 **10.8 MJy beam$^{-1}$**。2025 年最强事件的两个强细通道取平均并校正 2 s 时间平均后，在 1000 km 处约为 **36.0 MJy beam$^{-1}$**，约为本文 Event 1 的 3.9 倍（5.9 dB）。三者处于同一数量级。

Event 1 的谱 EIRP 约为 **$1.16\times10^{-6}$ W Hz$^{-1}$**，在 41.667 kHz 带宽内的包时段平均 EIRP 约为 **48.3 mW**（$-13.16$ dBW）。该数值直接继承 E08 理想旁瓣校正的系统误差，并且是沿 21CMA 方向推得的等效全向辐射功率，不能解释为卫星发射机的实际功率。

2023 和 2025 年的时间修正都依赖‘对方包长与 Event 1 相同’的假设。EDA2 成像没有分辨真实包边界，而且包长可能随 payload 长度和 PHY 配置变化；若数据包跨越成像时间片，单个时间片还会漏掉部分信号。因此这些是用于数量级对照的条件换算，不是精密交叉定标。

## 10. Grigg et al. (2025) 公开目录中的 137.05 MHz 发射片段 orbit phases分析

本节重新从公开原始 CSV 中筛选所有具有 137.05 MHz 窄带谱形的记录。对每个时间、NORAD ID 和极化组合的 31 个细通道进行与 Grigg et al. (2025) 相同的零基线一维 Gaussian 拟合，并严格采用其公开判据：$\sigma<28.9$ kHz 且 $R^2>0.95$ 时归为窄带。随后再从窄带结果中选取拟合中心位于 $137.055\pm0.030$ MHz 的目标记录。由于同一次小于 2 s 的发射可能跨越相邻的 2 s 成像积分，代码先合并同一时刻的两个极化，再将同一卫星相隔不超过 10 s 的相邻图像记录归为一个发射片段。

In [ ]:
# 从公开原始CSV独立筛选并汇总全部137.05 MHz窄带发射片段。
from scipy.optimize import curve_fit

group_columns = ['timestamp_unix_s', 'norad_id', 'polarisation_index']
catalogue_chunks = []
for chunk in pd.read_csv(
    GRIGG_CATALOGUE_FILE, sep=r'\s+', header=None, names=COLUMN_NAMES,
    chunksize=500_000,
):
    channel_index = chunk['fine_channel_index'].astype(int)
    band_lower = chunk['coarse_centre_hz'] - 15.0 * FINE_CHANNEL_WIDTH_HZ
    band_upper = chunk['coarse_centre_hz'] + 16.0 * FINE_CHANNEL_WIDTH_HZ
    keep = (
        channel_index.between(0, STACKED_CHANNEL_INDEX)
        & band_lower.le(TARGET_FREQUENCY_HZ)
        & band_upper.ge(TARGET_FREQUENCY_HZ)
    )
    if keep.any():
        catalogue_chunks.append(chunk.loc[keep].copy())

catalogue_rows = pd.concat(catalogue_chunks, ignore_index=True)
catalogue_rows['fine_channel_index'] = catalogue_rows['fine_channel_index'].astype(int)
catalogue_rows['fine_centre_hz'] = (
    catalogue_rows['coarse_centre_hz']
    + (catalogue_rows['fine_channel_index'] - 14.5) * FINE_CHANNEL_WIDTH_HZ
)

def zero_baseline_gaussian(frequency_hz, amplitude, centre_hz, sigma_hz):
    return amplitude * np.exp(-0.5 * ((frequency_hz - centre_hz) / sigma_hz) ** 2)

def fit_137p05_profile(group):
    fine = group[group['fine_channel_index'].between(0, N_USABLE_FINE_CHANNELS - 1)]
    x = fine['fine_centre_hz'].to_numpy(float)
    y = fine['flux_density_jy_beam'].to_numpy(float)
    if len(fine) != N_USABLE_FINE_CHANNELS or not np.all(np.isfinite(y)):
        return None
    peak_index = int(np.argmax(y))
    try:
        parameters, _ = curve_fit(
            zero_baseline_gaussian, x, y,
            p0=[max(float(y[peak_index]), 0.0), float(x[peak_index]), FINE_CHANNEL_WIDTH_HZ],
            bounds=([0.0, x.min(), 1.0], [np.inf, x.max(), 10.0 * FINE_CHANNEL_WIDTH_HZ]),
            maxfev=20_000,
        )
    except (RuntimeError, ValueError):
        return None
    model = zero_baseline_gaussian(x, *parameters)
    denominator = float(np.sum((y - y.mean()) ** 2))
    r_squared = 1.0 - float(np.sum((y - model) ** 2)) / denominator if denominator > 0 else np.nan
    return pd.Series({
        'fit_amplitude_jy_beam': parameters[0],
        'fit_centre_hz': parameters[1],
        'fit_sigma_hz': parameters[2],
        'fit_r_squared': r_squared,
    })

profile_fits = (
    catalogue_rows.groupby(group_columns, sort=False)
    .apply(fit_137p05_profile, include_groups=False)
    .dropna()
    .reset_index()
)
# 严格复现Grigg et al. (2025)公开的窄带分类判据。
narrowband_fits = profile_fits[
    profile_fits['fit_r_squared'].gt(0.95)
    & profile_fits['fit_sigma_hz'].lt(FINE_CHANNEL_WIDTH_HZ)
].copy()

# 在窄带分类完成后，再单独选取137.055 MHz附近的目标发射。
candidate_fits = narrowband_fits[
    narrowband_fits['fit_centre_hz'].sub(TARGET_FREQUENCY_HZ).abs().lt(30_000.0)
].copy()

stacked_rows = catalogue_rows[catalogue_rows['fine_channel_index'].eq(STACKED_CHANNEL_INDEX)].copy()
candidate_frames = candidate_fits.merge(
    stacked_rows, on=group_columns, how='left', validate='one_to_one',
)
candidate_frames['norad_id'] = candidate_frames['norad_id'].astype(int)
candidate_frames['polarisation_index'] = candidate_frames['polarisation_index'].astype(int)
candidate_timestamp_ms = (candidate_frames['timestamp_unix_s'] * 1000.0).round().astype('int64')
candidate_frames['utc'] = pd.to_datetime(candidate_timestamp_ms, unit='ms', utc=True)
candidate_frames['stacked_flux_mjy_beam'] = candidate_frames['flux_density_jy_beam'] / 1e6


# 同一时刻的两个极化先合并为一个图像采样点。
SEGMENT_MAX_GAP_S = 10.0
candidate_frames['fit_centre_mhz'] = candidate_frames['fit_centre_hz'] / 1e6


def polarisation_list(values):
    return ','.join(str(value) for value in sorted(set(values)))


image_samples = (
    candidate_frames.groupby(['norad_id', 'timestamp_unix_s'], as_index=False)
    .agg(
        utc=('utc', 'first'),
        detected_polarisations=('polarisation_index', polarisation_list),
        number_of_polarisations=('polarisation_index', 'nunique'),
        maximum_stacked_flux_mjy_beam=('stacked_flux_mjy_beam', 'max'),
        mean_fitted_centre_mhz=('fit_centre_mhz', 'mean'),
        median_fitted_sigma_khz=('fit_sigma_hz', lambda x: float(np.median(x)) / 1e3),
        tle_range_km=('tle_range_km', 'median'),
        tle_ra_deg=('tle_ra_deg', 'median'),
        tle_dec_deg=('tle_dec_deg', 'median'),
    )
    .sort_values(['norad_id', 'utc'])
    .reset_index(drop=True)
)

# 同一次短发射可能跨越相邻的2-s图像；10 s明显短于下一次独立候选。
time_gap_s = image_samples.groupby('norad_id')['utc'].diff().dt.total_seconds()
image_samples['time_from_previous_image_s'] = time_gap_s
image_samples['new_segment'] = time_gap_s.isna() | time_gap_s.gt(SEGMENT_MAX_GAP_S)
image_samples['segment_number'] = (
    image_samples.groupby('norad_id')['new_segment'].cumsum().astype(int)
)
segment_keys = ['norad_id', 'segment_number']
representative_index = image_samples.groupby(segment_keys)[
    'maximum_stacked_flux_mjy_beam'
].idxmax()
representative = image_samples.loc[
    representative_index,
    segment_keys + [
        'utc', 'tle_range_km', 'tle_ra_deg', 'tle_dec_deg',
        'maximum_stacked_flux_mjy_beam', 'mean_fitted_centre_mhz',
    ],
].rename(columns={
    'utc': 'representative_utc',
    'tle_range_km': 'representative_tle_range_km',
    'tle_ra_deg': 'representative_tle_ra_deg',
    'tle_dec_deg': 'representative_tle_dec_deg',
    'maximum_stacked_flux_mjy_beam': 'segment_maximum_stacked_flux_mjy_beam',
    'mean_fitted_centre_mhz': 'representative_fitted_centre_mhz',
})

emission_segments = (
    image_samples.groupby(segment_keys, as_index=False)
    .agg(
        segment_start_utc=('utc', 'min'),
        segment_end_utc=('utc', 'max'),
        number_of_image_samples=('utc', 'size'),
        detected_polarisations=(
            'detected_polarisations',
            lambda x: ','.join(sorted(set(','.join(x).split(',')))),
        ),
    )
    .merge(representative, on=segment_keys, how='left', validate='one_to_one')
    .sort_values(['segment_start_utc', 'norad_id'])
    .reset_index(drop=True)
)
emission_segments.insert(0, 'segment_id', np.arange(1, len(emission_segments) + 1))
emission_segments['image_time_span_s'] = (
    emission_segments['segment_end_utc'] - emission_segments['segment_start_utc']
).dt.total_seconds()
emission_segments['observing_date'] = emission_segments['segment_start_utc'].dt.date

satellite_summary = (
    emission_segments.groupby('norad_id', as_index=False)
    .agg(
        number_of_segments=('segment_id', 'size'),
        number_of_observing_dates=('observing_date', 'nunique'),
        first_segment_utc=('segment_start_utc', 'min'),
        last_segment_utc=('segment_end_utc', 'max'),
        maximum_stacked_flux_mjy_beam=(
            'segment_maximum_stacked_flux_mjy_beam', 'max'
        ),
        minimum_tle_range_km=('representative_tle_range_km', 'min'),
        median_tle_range_km=('representative_tle_range_km', 'median'),
        maximum_tle_range_km=('representative_tle_range_km', 'max'),
    )
    .sort_values(['number_of_segments', 'norad_id'], ascending=[False, True])
    .reset_index(drop=True)
)

candidate_output = OUTPUT_DIR / 'grigg2025_137p05_candidate_frames.csv'
image_output = OUTPUT_DIR / 'grigg2025_137p05_image_samples.csv'
segment_output = OUTPUT_DIR / 'grigg2025_137p05_emission_segments.csv'
summary_output = OUTPUT_DIR / 'grigg2025_137p05_satellite_summary.csv'
candidate_frames.to_csv(candidate_output, index=False)
image_samples.to_csv(image_output, index=False)
emission_segments.to_csv(segment_output, index=False)
satellite_summary.to_csv(summary_output, index=False)

print('137.05-MHz selection summary')
print(f'  Fine-channel profiles successfully fitted: {len(profile_fits)}')
print(f'  Narrowband rows under the published Grigg et al. criteria: {len(narrowband_fits)}')
print(f'  Fine-profile candidate rows (polarisation retained): {len(candidate_frames)}')
print(f'  Unique satellite/image times: {len(image_samples)}')
print(f'  Merged emission segments: {len(emission_segments)}')
print(f'  Unique NORAD objects: {emission_segments["norad_id"].nunique()}')
print(f'  One-image segments: {(emission_segments["number_of_image_samples"] == 1).sum()}')
print(f'  Two-image segments: {(emission_segments["number_of_image_samples"] == 2).sum()}')
print('\nAll merged 137.05-MHz emission segments:')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(emission_segments)
print('\nSummary by NORAD object:')
display(satellite_summary)
print('Saved:', candidate_output)
print('Saved:', image_output)
print('Saved:', segment_output)
print('Saved:', summary_output)


### 10.1 第一阶段输出说明

代码保存四个层级的结果：`candidate_frames`保留每个极化的谱形筛选结果；`image_samples`合并同一卫星、同一时刻的两个极化；`emission_segments`进一步合并可能属于同一次发射的相邻2 s图像；`satellite_summary`按NORAD目标汇总片段数量、观测日期、时间范围、最大强度和斜距范围。后续轨道分析应以`emission_segments`中的代表时刻和NORAD ID为入口，而不应把两个极化或跨越相邻成像积分的记录重复计算。

### 10.2 卫星—观测日期组合

为后续判断发射时卫星处于升轨、稳定运行还是降轨阶段，将合并后的发射片段按NORAD ID和UTC日期归并。表中保留每个组合的片段数、首末时刻及全部代表时刻；轨道阶段的统计单位为“卫星—日期”，但精确UTC仍保留供TLE匹配和核查。

In [ ]:
# 输出全部137.05 MHz发射片段对应的卫星—UTC日期组合。
def join_utc_times(values):
    times = sorted(pd.to_datetime(values, utc=True))
    return '; '.join(pd.Timestamp(value).isoformat().replace('+00:00', 'Z') for value in times)

satellite_date_combinations = (
    emission_segments.groupby(['norad_id', 'observing_date'], as_index=False)
    .agg(
        number_of_emission_segments=('segment_id', 'size'),
        first_segment_utc=('segment_start_utc', 'min'),
        last_segment_utc=('segment_end_utc', 'max'),
        representative_times_utc=('representative_utc', join_utc_times),
        maximum_stacked_flux_mjy_beam=('segment_maximum_stacked_flux_mjy_beam', 'max'),
        minimum_tle_range_km=('representative_tle_range_km', 'min'),
        maximum_tle_range_km=('representative_tle_range_km', 'max'),
    )
    .sort_values(['observing_date', 'norad_id'])
    .reset_index(drop=True)
)
satellite_date_combinations.insert(
    0, 'combination_id', np.arange(1, len(satellite_date_combinations) + 1)
)

satellite_date_output = OUTPUT_DIR / 'grigg2025_137p05_satellite_date_combinations.csv'
satellite_date_combinations.to_csv(satellite_date_output, index=False)

print('Satellite-date combinations for subsequent orbit analysis')
print(f'  Unique NORAD objects: {satellite_date_combinations["norad_id"].nunique()}')
print(f'  Satellite-date combinations: {len(satellite_date_combinations)}')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(satellite_date_combinations)
print('Saved:', satellite_date_output)


### 10.3 Space-Track历史GP/TLE下载完整性检查

本单元只核查下载结果，不计算轨道高度变化率，也不对卫星进行升轨、稳定运行或降轨分类。检查包括查询时间窗、NORAD目标是否齐全、重复发布的历史记录、关键字段缺失、TLE行长和校验和，以及每个“卫星—日期”组合在观测前后的GP历元覆盖。如果观测前或观测后没有可用历元，表中会单独标记。

In [ ]:
# 只检查Space-Track历史GP/TLE数据的下载与时间覆盖完整性。
SPACETRACK_DIR = DATA_DIR / 'external' / 'spacetrack' / 'all_137p05_emitters'
QUERY_PLAN_FILE = SPACETRACK_DIR / 'spacetrack_query_plan.csv'
GP_HISTORY_FILE = SPACETRACK_DIR / 'grigg2025_all_137p05_gp_history.csv'
UNIQUE_GP_FILE = SPACETRACK_DIR / 'grigg2025_all_137p05_unique_gp_epochs.csv'
COVERAGE_FILE = SPACETRACK_DIR / 'grigg2025_all_137p05_gp_coverage.csv'
SATELLITE_DATE_FILE = OUTPUT_DIR / 'grigg2025_137p05_satellite_date_combinations.csv'

for required_file in (QUERY_PLAN_FILE, GP_HISTORY_FILE, UNIQUE_GP_FILE, COVERAGE_FILE, SATELLITE_DATE_FILE):
    if not required_file.is_file():
        raise FileNotFoundError(required_file)

query_plan = pd.read_csv(QUERY_PLAN_FILE)
gp_history = pd.read_csv(GP_HISTORY_FILE, low_memory=False)
unique_gp = pd.read_csv(UNIQUE_GP_FILE, low_memory=False)
download_coverage = pd.read_csv(COVERAGE_FILE)
download_combinations = pd.read_csv(SATELLITE_DATE_FILE)

for table in (gp_history, unique_gp):
    table['NORAD_CAT_ID'] = table['NORAD_CAT_ID'].astype(int)
    table['EPOCH'] = pd.to_datetime(table['EPOCH'], utc=True, format='mixed')
    table['CREATION_DATE'] = pd.to_datetime(table['CREATION_DATE'], utc=True, format='mixed')
download_combinations['norad_id'] = download_combinations['norad_id'].astype(int)
download_combinations['observing_date'] = pd.to_datetime(
    download_combinations['observing_date'], utc=True
).dt.normalize()
for column in ('first_segment_utc', 'last_segment_utc'):
    download_combinations[column] = pd.to_datetime(
        download_combinations[column], utc=True, format='mixed'
    )

def tle_checksum_valid(line):
    if not isinstance(line, str) or len(line) != 69 or not line[68].isdigit():
        return False
    checksum = sum(
        int(character) if character.isdigit() else 1 if character == '-' else 0
        for character in line[:68]
    ) % 10
    return checksum == int(line[68])

requested_ids = set(download_combinations['norad_id'])
downloaded_ids = set(unique_gp['NORAD_CAT_ID'])
duplicate_release_rows = len(gp_history) - len(unique_gp)
line1_valid = unique_gp['TLE_LINE1'].map(tle_checksum_valid)
line2_valid = unique_gp['TLE_LINE2'].map(tle_checksum_valid)

download_audit_summary = pd.DataFrame({
    'check': [
        'Merged query groups',
        'Requested satellite-date combinations',
        'Requested unique NORAD objects',
        'Downloaded unique NORAD objects',
        'Raw GP_HISTORY rows',
        'Unique NORAD+EPOCH rows',
        'Repeated historical releases removed',
        'Valid TLE line 1 checksums',
        'Valid TLE line 2 checksums',
    ],
    'value': [
        len(query_plan), len(download_combinations), len(requested_ids),
        len(downloaded_ids), len(gp_history), len(unique_gp),
        duplicate_release_rows, int(line1_valid.sum()), int(line2_valid.sum()),
    ],
})

important_fields = [
    'NORAD_CAT_ID', 'OBJECT_NAME', 'EPOCH', 'CREATION_DATE',
    'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE',
    'ARG_OF_PERICENTER', 'MEAN_ANOMALY', 'BSTAR', 'MEAN_MOTION_DOT',
    'SEMIMAJOR_AXIS', 'TLE_LINE1', 'TLE_LINE2',
]
field_completeness = pd.DataFrame({
    'field': important_fields,
    'non_missing_rows': [int(unique_gp[field].notna().sum()) for field in important_fields],
    'missing_rows': [int(unique_gp[field].isna().sum()) for field in important_fields],
})

PADDING_DAYS = 14
coverage_rows = []
for combination in download_combinations.itertuples(index=False):
    window_start = combination.observing_date - pd.Timedelta(days=PADDING_DAYS)
    window_end = combination.observing_date + pd.Timedelta(days=PADDING_DAYS + 1)
    satellite_gp = unique_gp[
        unique_gp['NORAD_CAT_ID'].eq(combination.norad_id)
        & unique_gp['EPOCH'].ge(window_start)
        & unique_gp['EPOCH'].lt(window_end)
    ].sort_values('EPOCH')
    before = satellite_gp[satellite_gp['EPOCH'].le(combination.first_segment_utc)]
    after = satellite_gp[satellite_gp['EPOCH'].ge(combination.last_segment_utc)]
    preceding_epoch = before['EPOCH'].max() if not before.empty else pd.NaT
    following_epoch = after['EPOCH'].min() if not after.empty else pd.NaT

    if satellite_gp.empty:
        object_name = None
        launch_date = None
        invalid_tle_rows = 0
        maximum_epoch_gap_hours = np.nan
        status = 'NO GP RECORDS'
    else:
        object_name = satellite_gp['OBJECT_NAME'].dropna().iloc[0]
        launch_values = satellite_gp['LAUNCH_DATE'].dropna()
        launch_date = launch_values.iloc[0] if not launch_values.empty else None
        valid_tle_rows = (
            satellite_gp['TLE_LINE1'].map(tle_checksum_valid)
            & satellite_gp['TLE_LINE2'].map(tle_checksum_valid)
        )
        invalid_tle_rows = int((~valid_tle_rows).sum())
        maximum_epoch_gap_hours = (
            satellite_gp['EPOCH'].diff().dt.total_seconds().div(3600).max()
        )
        if pd.isna(preceding_epoch):
            status = 'NO PRE-OBSERVATION GP'
        elif pd.isna(following_epoch):
            status = 'NO POST-OBSERVATION GP'
        elif invalid_tle_rows:
            status = 'INVALID TLE CHECKSUM'
        else:
            status = 'OK: OBSERVATION BRACKETED'

    coverage_rows.append({
        'combination_id': combination.combination_id,
        'norad_id': combination.norad_id,
        'object_name': object_name,
        'launch_date': launch_date,
        'observing_date': combination.observing_date.date().isoformat(),
        'number_of_unique_gp_epochs': len(satellite_gp),
        'first_gp_epoch': satellite_gp['EPOCH'].min() if not satellite_gp.empty else pd.NaT,
        'last_gp_epoch': satellite_gp['EPOCH'].max() if not satellite_gp.empty else pd.NaT,
        'preceding_gp_epoch': preceding_epoch,
        'preceding_epoch_gap_hours': (
            (combination.first_segment_utc - preceding_epoch).total_seconds() / 3600
            if pd.notna(preceding_epoch) else np.nan
        ),
        'following_gp_epoch': following_epoch,
        'following_epoch_gap_hours': (
            (following_epoch - combination.last_segment_utc).total_seconds() / 3600
            if pd.notna(following_epoch) else np.nan
        ),
        'maximum_epoch_gap_hours': maximum_epoch_gap_hours,
        'invalid_tle_rows': invalid_tle_rows,
        'coverage_status': status,
    })

gp_coverage_audit = pd.DataFrame(coverage_rows)
missing_requested_ids = sorted(requested_ids - downloaded_ids)
unexpected_downloaded_ids = sorted(downloaded_ids - requested_ids)

print('Space-Track GP_HISTORY download audit')
display(download_audit_summary)
print('Query plan:')
display(query_plan)
print('Important-field completeness in the unique-epoch table:')
display(field_completeness)
print('Missing requested NORAD IDs:', missing_requested_ids or 'None')
print('Unexpected downloaded NORAD IDs:', unexpected_downloaded_ids or 'None')
print('Coverage around every satellite-date observation:')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(gp_coverage_audit)
print('Coverage-status counts:')
display(gp_coverage_audit['coverage_status'].value_counts().rename_axis('status').to_frame('count'))

coverage_warnings = gp_coverage_audit[
    gp_coverage_audit['coverage_status'].ne('OK: OBSERVATION BRACKETED')
]
if coverage_warnings.empty:
    print('All satellite-date combinations have valid GP epochs on both sides of the observations.')
else:
    print('Combinations requiring attention (this does not by itself imply a failed download):')
    display(coverage_warnings)


### 10.4 每个卫星—日期组合的轨道高度变化指标

为判断137.05 MHz发射是否仅出现在升轨阶段，首先对每个“卫星—日期”组合计算连续的轨道高度变化指标，本步不进行轨道阶段分类。Space-Track给出的平均半长轴通过 $h_{\rm mean}=a-6378.137~\mathrm{km}$ 转换为平均轨道高度。同一天内的多个GP历元先取高度中位数，以避免TLE更新频率不同对趋势拟合产生权重差异。

对每个组合，代码使用当日首个和最后一个发射片段之间的中点作为日尺度的参考时刻，计算该时刻前后最近GP历元之间的有限差分变化率，并对检测日前后 $\pm14$ d窗口内的逐日中位高度作Theil--Sen稳健线性拟合。同时输出窗口内的有效日数和双边覆盖状态，因此单边数据不会被误当作与完整样本等价。

In [ ]:
# 计算逐日平均轨道高度和检测日前后14天的稳健趋势，本步不分类轨道阶段。
from scipy.stats import theilslopes

EARTH_EQUATORIAL_RADIUS_KM = 6378.137
TREND_HALF_WINDOWS_DAYS = (14,)

orbit_gp = unique_gp[
    unique_gp['NORAD_CAT_ID'].isin(download_combinations['norad_id'])
].copy()
orbit_gp['mean_altitude_km'] = (
    pd.to_numeric(orbit_gp['SEMIMAJOR_AXIS'], errors='coerce')
    - EARTH_EQUATORIAL_RADIUS_KM
)
orbit_gp['epoch_date_utc'] = orbit_gp['EPOCH'].dt.floor('D')

daily_mean_altitude = (
    orbit_gp.groupby(['NORAD_CAT_ID', 'epoch_date_utc'], as_index=False)
    .agg(
        number_of_gp_epochs=('EPOCH', 'size'),
        first_gp_epoch=('EPOCH', 'min'),
        last_gp_epoch=('EPOCH', 'max'),
        median_mean_altitude_km=('mean_altitude_km', 'median'),
        minimum_mean_altitude_km=('mean_altitude_km', 'min'),
        maximum_mean_altitude_km=('mean_altitude_km', 'max'),
    )
    .rename(columns={'NORAD_CAT_ID': 'norad_id'})
    .sort_values(['norad_id', 'epoch_date_utc'])
    .reset_index(drop=True)
)


def robust_daily_altitude_slope(satellite_daily, reference_utc, half_window_days):
    reference_day = reference_utc.floor('D')
    lower = reference_day - pd.Timedelta(days=half_window_days)
    upper = reference_day + pd.Timedelta(days=half_window_days)
    selected = satellite_daily[
        satellite_daily['epoch_date_utc'].between(lower, upper)
    ].copy()
    selected = selected.dropna(subset=['median_mean_altitude_km'])
    before_count = int((selected['epoch_date_utc'] < reference_day).sum())
    after_count = int((selected['epoch_date_utc'] > reference_day).sum())
    if before_count and after_count:
        coverage = 'TWO-SIDED'
    elif before_count:
        coverage = 'PRE-ONLY'
    elif after_count:
        coverage = 'POST-ONLY'
    else:
        coverage = 'REFERENCE-DAY ONLY' if len(selected) else 'NO DATA'
    if len(selected) < 3:
        slope = np.nan
    else:
        x_days = (
            selected['epoch_date_utc'] - reference_day
        ).dt.total_seconds().to_numpy() / 86400.0
        y_km = selected['median_mean_altitude_km'].to_numpy(float)
        slope = float(theilslopes(y_km, x_days).slope)
    return {
        'slope_km_per_day': slope,
        'number_of_days': len(selected),
        'number_of_pre_days': before_count,
        'number_of_post_days': after_count,
        'coverage': coverage,
    }


trend_rows = []
for combination in download_combinations.itertuples(index=False):
    reference_utc = combination.first_segment_utc + (
        combination.last_segment_utc - combination.first_segment_utc
    ) / 2
    satellite_gp = orbit_gp[
        orbit_gp['NORAD_CAT_ID'].eq(combination.norad_id)
    ].sort_values('EPOCH')
    satellite_daily = daily_mean_altitude[
        daily_mean_altitude['norad_id'].eq(combination.norad_id)
    ].sort_values('epoch_date_utc')
    object_name = satellite_gp['OBJECT_NAME'].dropna().iloc[0] if not satellite_gp.empty else None
    launch_values = satellite_gp['LAUNCH_DATE'].dropna()
    launch_date = launch_values.iloc[0] if not launch_values.empty else None

    preceding = satellite_gp[satellite_gp['EPOCH'].le(reference_utc)].tail(1)
    following = satellite_gp[satellite_gp['EPOCH'].ge(reference_utc)].head(1)
    preceding_epoch = preceding['EPOCH'].iloc[0] if not preceding.empty else pd.NaT
    following_epoch = following['EPOCH'].iloc[0] if not following.empty else pd.NaT
    preceding_altitude = (
        float(preceding['mean_altitude_km'].iloc[0]) if not preceding.empty else np.nan
    )
    following_altitude = (
        float(following['mean_altitude_km'].iloc[0]) if not following.empty else np.nan
    )
    if not preceding.empty and not following.empty and following_epoch > preceding_epoch:
        bracket_duration_days = (
            following_epoch - preceding_epoch
        ).total_seconds() / 86400.0
        bracket_rate = (following_altitude - preceding_altitude) / bracket_duration_days
        interpolation_fraction = (
            (reference_utc - preceding_epoch).total_seconds()
            / (following_epoch - preceding_epoch).total_seconds()
        )
        interpolated_altitude = (
            preceding_altitude
            + interpolation_fraction * (following_altitude - preceding_altitude)
        )
        bracket_status = 'BRACKETED'
    else:
        bracket_duration_days = np.nan
        bracket_rate = np.nan
        interpolated_altitude = np.nan
        bracket_status = (
            'NO PRE-OBSERVATION GP' if preceding.empty
            else 'NO POST-OBSERVATION GP' if following.empty
            else 'INVALID BRACKET'
        )

    row = {
        'combination_id': combination.combination_id,
        'norad_id': combination.norad_id,
        'object_name': object_name,
        'launch_date': launch_date,
        'observing_date': combination.observing_date.date().isoformat(),
        'observation_reference_utc': reference_utc,
        'preceding_gp_epoch': preceding_epoch,
        'preceding_mean_altitude_km': preceding_altitude,
        'preceding_epoch_gap_hours': (
            (reference_utc - preceding_epoch).total_seconds() / 3600.0
            if pd.notna(preceding_epoch) else np.nan
        ),
        'following_gp_epoch': following_epoch,
        'following_mean_altitude_km': following_altitude,
        'following_epoch_gap_hours': (
            (following_epoch - reference_utc).total_seconds() / 3600.0
            if pd.notna(following_epoch) else np.nan
        ),
        'bracket_duration_days': bracket_duration_days,
        'bracket_altitude_rate_km_per_day': bracket_rate,
        'interpolated_mean_altitude_km': interpolated_altitude,
        'bracket_status': bracket_status,
    }
    for half_window_days in TREND_HALF_WINDOWS_DAYS:
        result = robust_daily_altitude_slope(
            satellite_daily, reference_utc, half_window_days
        )
        prefix = f'trend_{half_window_days}d'
        row[f'{prefix}_km_per_day'] = result['slope_km_per_day']
        row[f'{prefix}_number_of_days'] = result['number_of_days']
        row[f'{prefix}_pre_days'] = result['number_of_pre_days']
        row[f'{prefix}_post_days'] = result['number_of_post_days']
        row[f'{prefix}_coverage'] = result['coverage']
    trend_rows.append(row)

orbit_trend_metrics = pd.DataFrame(trend_rows).sort_values(
    ['observing_date', 'norad_id']
).reset_index(drop=True)

daily_altitude_output = OUTPUT_DIR / 'grigg2025_137p05_daily_mean_orbit_altitude.csv'
trend_metrics_output = OUTPUT_DIR / 'grigg2025_137p05_orbit_trend_metrics.csv'
daily_mean_altitude.to_csv(daily_altitude_output, index=False)
orbit_trend_metrics.to_csv(trend_metrics_output, index=False)

print(f'Daily mean-orbit altitude rows: {len(daily_mean_altitude)}')
print('First 20 daily rows (the complete table is saved to CSV):')
display(daily_mean_altitude.head(20))
print('Orbit-trend metrics for all satellite-date combinations:')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(orbit_trend_metrics)
print('Bracket-status counts:')
display(orbit_trend_metrics['bracket_status'].value_counts().rename_axis('status').to_frame('count'))
print('Saved:', daily_altitude_output)
print('Saved:', trend_metrics_output)


### 10.5 从首次检测前90天至当前的完整轨道演化与初步分类

本节读取逐颗卫星下载并去除重复历元后的完整Space-Track GP历史，将75个发射片段归并为28个“卫星—UTC日期”组合。对每个组合重新计算事件时平均轨道高度，以及检测日前后$\pm14$ d的Theil--Sen稳健高度变化率；同时利用事件后30--90 d的轨道高度检查局部平坦阶段是否只是升轨过程中的暂时停留。

初步分类只使用$\pm14$ d趋势：$|\dot{h}|\leq0.2$ km d$^{-1}$时归为稳定轨道阶段，$\dot{h}>0.2$ km d$^{-1}$时归为升轨，$\dot{h}<-0.2$ km d$^{-1}$时归为降轨。所有连续指标和数据覆盖状态均保留在输出表中，分类结果需要结合随后生成的逐颗卫星长期轨道图进行人工复核；该分类描述的是发射日期附近的总体轨道阶段，不用于确定秒级发射是否对应某次具体轨道机动。


In [ ]:
from IPython.display import Image

ORBIT_LIFECYCLE_SCRIPT = (
    NOTEBOOK_DIR / 'analyse_spacetrack_full_history_137p05_emitters.py'
)
FULL_HISTORY_FILE = (
    DATA_DIR / 'external' / 'spacetrack'
    / 'full_137p05_emitters' / 'grigg2025_full_137p05_unique_gp_epochs.csv'
)
EMISSION_SEGMENT_FILE = (
    OUTPUT_DIR / 'grigg2025_137p05_emission_segments.csv'
)
for required in (ORBIT_LIFECYCLE_SCRIPT, FULL_HISTORY_FILE, EMISSION_SEGMENT_FILE):
    if not required.is_file():
        raise FileNotFoundError(required)

subprocess.run(
    [
        sys.executable, str(ORBIT_LIFECYCLE_SCRIPT),
        '--history', str(FULL_HISTORY_FILE),
        '--segments', str(EMISSION_SEGMENT_FILE),
        '--output-dir', str(OUTPUT_DIR),
        '--stable-rate-threshold', '0.2',
        '--plots-per-page', '4',
    ],
    check=True,
    cwd=NOTEBOOK_DIR,
)

phase_summary_file = OUTPUT_DIR / 'grigg2025_137p05_orbital_phase_summary.csv'
phase_summary = pd.read_csv(phase_summary_file)
display(
    phase_summary['preliminary_orbital_phase']
    .value_counts()
    .rename_axis('preliminary_orbital_phase')
    .to_frame('number_of_satellite_date_combinations')
)
display(phase_summary)
display(Image(filename=str(OUTPUT_DIR / 'grigg2025_137p05_orbit_lifecycle_page_01.png')))
print('Complete multi-page figure:', OUTPUT_DIR / 'grigg2025_137p05_orbit_lifecycle_panels.pdf')


### 10.6 发射时的轨道高度变化率与轨道状态汇总表

为便于直接检查，本表只保留卫星名称、NORAD编号、检测日期、发射时平均轨道高度、每天高度变化率和轨道状态。每天高度变化率采用观测日期前后$\pm14$ d逐日平均轨道高度的Theil--Sen稳健斜率，单位为km d$^{-1}$。同一卫星若在两个UTC日期被检测到，则分别列出；缺少事件前TLE的目标同时保留覆盖状态说明。


In [ ]:
ORBIT_STATE_TABLE_SCRIPT = NOTEBOOK_DIR / 'make_grigg2025_orbital_state_table.py'
ORBIT_STATE_TABLE_FILE = OUTPUT_DIR / 'grigg2025_137p05_orbital_state_table.csv'
subprocess.run(
    [
        sys.executable, str(ORBIT_STATE_TABLE_SCRIPT),
        '--input', str(phase_summary_file),
        '--output', str(ORBIT_STATE_TABLE_FILE),
    ],
    check=True,
    cwd=NOTEBOOK_DIR,
)
orbital_state_table = pd.read_csv(ORBIT_STATE_TABLE_FILE)
display(
    orbital_state_table[
        [
            'satellite_name', 'norad_id', 'detection_date_utc',
            'mean_altitude_at_detection_km',
            'altitude_change_rate_km_per_day', 'orbital_state',
        ]
    ]
)
print('Saved:', ORBIT_STATE_TABLE_FILE)
